In [34]:
# scrape_interconnection_projects.py
# ----------------------------------
import json, time, pathlib, traceback
from typing import List, Dict

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm              # nice progress bar (pip install tqdm)

# ----------------------------------------------------------------------
# 1) CONFIG -------------------------------------------------------------
# ----------------------------------------------------------------------
INPUT_XLSX   = "testdata/Interconnectionfyi_interconnection_queue_data_in_the_US_19962025.2025-8-5.File-4.xlsx"       # <-- your workbook
LINK_COLUMN  = "Project Details"                    # <-- column name that holds URLs
PAUSE_SEC    = 0.05                      # polite delay between requests
OUT_CSV      = "interconnection_projects_2025_Aug05_part4.csv"
#OUT_PARQUET  = "interconnection_projects.parquet"

# ----------------------------------------------------------------------
# 2) HELPER: fetch details for ONE project -----------------------------
# ----------------------------------------------------------------------
def fetch_project(url: str) -> Dict[str, str]:
    """Return the metadata dict for a single interconnection project page."""
    html = requests.get(url, timeout=15).text
    soup = BeautifulSoup(html, "html.parser")

    script_tag = soup.find("script", id="__NEXT_DATA__")
    if script_tag is None:                          # fallback: scrape table
        rows = soup.select("table tr")
        return {
            r.find_all("td")[0].get_text(strip=True): 
            r.find_all("td")[1].get_text(" ", strip=True)
            for r in rows if r.find_all("td")
        }

    payload = json.loads(script_tag.string)
    return payload["props"]["pageProps"]["serializedProjectProps"]["json"]


In [35]:
df = pd.read_excel(INPUT_XLSX)
df[LINK_COLUMN] = df[LINK_COLUMN].apply(lambda x: x.split("|")[1] if isinstance(x, str) and "|" in x else None)
links = (df[LINK_COLUMN].dropna().unique())

records: List[Dict[str, str]] = []
errors  : List[str]           = []

for link in tqdm(links, desc="Scraping projects"):
    try:
        data = fetch_project(link)
        data["source_url"] = link          # keep a trace
        records.append(data)
    except Exception:
        errors.append(link)
        traceback.print_exc()
    time.sleep(PAUSE_SEC)

# assemble dataframe
df = pd.DataFrame(records)
print(f"\n✓ scraped {len(df)} projects "
        f"(failed: {len(errors)})")

# save
df.to_csv(OUT_CSV, index=False)
#df.to_parquet(OUT_PARQUET, index=False)

if errors:
    pathlib.Path("failed_links.txt").write_text("\n".join(errors))
    print("⚠ some links failed → see failed_links.txt")

Scraping projects:  73%|███████▎  | 7069/9707 [1:19:56<27:16,  1.61it/s]  Traceback (most recent call last):
  File "/Users/tonyliu/Documents/DriftNet/data_centers/driftnet/lib/python3.9/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/Users/tonyliu/Documents/DriftNet/data_centers/driftnet/lib/python3.9/site-packages/urllib3/connection.py", line 516, in getresponse
    httplib_response = super().getresponse()
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/http/client.py", line 1349, in getresponse
    response.begin()
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/http/client.py", line 316, in begin
    version, status, reason = self._read_status()
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/http/client.py", line 277, in _read_status
    lin


✓ scraped 9706 projects (failed: 1)
⚠ some links failed → see failed_links.txt


In [45]:
df_test0 = pd.read_csv("interconnection_projects_2025_Aug05.csv")
df_test = pd.read_csv("interconnection_projects_2025_Aug05_part2.csv")
df_test2 = pd.read_csv("interconnection_projects_2025_Aug05_part3.csv")
df_test3 = pd.read_csv("interconnection_projects_2025_Aug05_part4.csv")
df_test = pd.concat([df_test0, df_test, df_test2, df_test3], ignore_index=True)
df_test.head()

,Actual Completion Date,Canonical Generation Types,Canonical Transmission Owners,Capacity Range (MW),County,Interconnecting Entity,Interconnection Location,Power Market,Project Name,Project Type,...,Queue Date,Queue ID,State,Status,Transmission Owner,Withdrawal Comment,Withdrawn Date,_uniqueId,utility,source_url
0,NaN,['Battery'],['Bonneville Power Administration'],0 - 10,Franklin County,NaN,Connell Tap on the Benton-Scooteney line.,West,NaN,Generation,...,2025-07-31T13:31:59.999Z,G1079,WA,Active,Bonneville Power Administration,.,NaN,bpa-g1079,NaN,https://www.interconnection.fyi/project/bpa-g1...
1,NaN,[],['Bonneville Power Administration'],500 - 750,Morrow County,Umatilla Electric Cooperative,Sixmile Canyon Substation at 230 kV,West,Steelhead 230 kV,Load,...,2025-07-31T12:37:00.000Z,L0661,OR,Active,Bonneville Power Administration,.,NaN,bpa-l0661,Umatilla Electric Cooperative,https://www.interconnection.fyi/project/bpa-l0...
2,NaN,['Battery'],['American Transmission Company'],200 - 300,Milwaukee County,NaN,Granville345kV Substation,MISO,NaN,Generation,...,2025-07-29T04:00:00.000Z,J3006,WI,Active,AMERICAN TRANSMISSION COMPANY,NaN,NaN,miso-j3006,AMERICAN TRANSMISSION COMPANY,https://www.interconnection.fyi/project/miso-j...
3,NaN,['Solar'],['Michigan Electric Transmission Company'],200 - 300,Calhoun County,NaN,MAJESTIC - ONEIDA 345.0kV,MISO,NaN,Generation,...,2025-07-29T04:00:00.000Z,J3050,MI,Active,METC,NaN,NaN,miso-j3050,METC,https://www.interconnection.fyi/project/miso-j...
4,NaN,['Battery'],[],0 - 10,San Patricio,Convergent Energy Solutions LLC,8566 HOT IRON 4A 138 kV\n8569 HOT IRON 4B 138 ...,ERCOT,Portland Storage SLF,Generation,...,2025-07-29T04:00:00.000Z,27INR0524,TX,Active,NaN,NaN,NaN,ercot-27inr0524,NaN,https://www.interconnection.fyi/project/ercot-...


In [47]:
df_test.to_csv("data/interconnection_projects_2025_Aug05_all.csv", index=False)

In [43]:
df_test.iloc[0]

Actual Completion Date                                                         NaN
Canonical Generation Types                                    ['Battery', 'Solar']
Canonical Transmission Owners                              ['Central Maine Power']
Capacity Range (MW)                                                         0 - 10
County                                                                 York County
Interconnecting Entity                                                         NaN
Interconnection Location                 CMP Kimball Road/Lovell 115 kV Substation
Power Market                                                                ISO-NE
Project Name                                                 Kimball DG Area Study
Project Type                                                            Generation
Proposed Completion Date                                  2028-06-30T04:00:00.000Z
Queue Date                                                2022-05-02T04:00:00.000Z
Queu